In [1]:
# !pip install torch
# !pip install torchvision
# !pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu129 -U


Looking in indexes: https://download.pytorch.org/whl/nightly/cu129


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Hyperparameters
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Recommended code block to check CUDA details ---

# Check if CUDA (GPU support) is available
if device == "cuda":
    # Get the number of available GPUs
    gpu_count = torch.cuda.device_count()
    print(f"CUDA is available. Using {gpu_count} GPU(s).")
    
    # Print the name of each GPU
    for i in range(gpu_count):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("CUDA not available. Using CPU.")

# --- End of recommended block ---

batch_size = 128
image_size = 64
channels_img = 1 # MNIST is grayscale
z_dim = 100 # Latent dimension (noise)
num_epochs = 50 # As required by the assignment

# Quick fix: Renamed 'transforms' variable to avoid conflict with the module
transform_pipeline = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5 for _ in range(channels_img)], [0.5 for _ in range(channels_img)]),
])

dataset = datasets.MNIST(root="dataset/", train=True, transform=transform_pipeline, download=True)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,num_workers=4 )

CUDA is available. Using 1 GPU(s).
  GPU 0: NVIDIA GeForce RTX 5060


In [3]:
class Generator(nn.Module):
    def __init__(self, z_dim, channels_img, features_g):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            # Input: N x z_dim x 1 x 1
            nn.ConvTranspose2d(z_dim, features_g * 16, 4, 1, 0, bias=False), # N x f_g*16 x 4 x 4
            nn.BatchNorm2d(features_g * 16),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 16, features_g * 8, 4, 2, 1, bias=False), # N x f_g*8 x 8 x 8
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 8, features_g * 4, 4, 2, 1, bias=False), # N x f_g*4 x 16 x 16
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1, bias=False), # N x f_g*2 x 32 x 32
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 2, channels_img, 4, 2, 1, bias=False), # N x channels_img x 64 x 64
            nn.Tanh() # Output: [-1, 1]
        )

    def forward(self, x):
        return self.net(x)

In [4]:
# (Keep the same architecture, just remove the final activation)
class Critic(nn.Module):
    def __init__(self, channels_img, features_d):
        super(Critic, self).__init__()
        self.net = nn.Sequential(
            # Input: N x channels_img x 64 x 64
            nn.Conv2d(channels_img, features_d, 4, 2, 1, bias=False), # N x f_d x 32 x 32
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias=False), # N x f_d*2 x 16 x 16
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 2, features_d * 4, 4, 2, 1, bias=False), # N x f_d*4 x 8 x 8
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 4, features_d * 8, 4, 2, 1, bias=False), # N x f_d*8 x 4 x 4
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 8, 1, 4, 2, 0, bias=False), # N x 1 x 1 x 1
        )

    def forward(self, x):
        return self.net(x)

In [5]:
lr = 5e-5 # Lower learning rate is common for WGAN
# batch_size, image_size, etc. remain the same
critic_iterations = 5 # Train the critic 5 times for every generator training
clip_value = 0.01 # The value for weight clipping

# Initialize models
gen = Generator(z_dim, channels_img, 64).to(device)
critic = Critic(channels_img, 64).to(device)

# Optimizers --> Use RMSprop
opt_gen = optim.RMSprop(gen.parameters(), lr=lr)
opt_critic = optim.RMSprop(critic.parameters(), lr=lr)

In [6]:
# Create a fixed noise vector to see the progression of the generator
fixed_noise = torch.randn(64, z_dim, 1, 1).to(device) # We'll generate a grid of 8x8=64 images

In [7]:
# Before the training loop starts
G_losses = []
D_losses = []

In [ ]:
# Main WGAN training loop
for epoch in range(num_epochs):
    for batch_idx, (real, _) in enumerate(loader):
        real = real.to(device)
        batch_size = real.shape[0]

        # Train Critic: max E[critic(real)] - E[critic(fake)]
        # equivalent to minimizing E[critic(fake)] - E[critic(real)]
        for _ in range(critic_iterations):
            noise = torch.randn(batch_size, z_dim, 1, 1).to(device)
            fake = gen(noise)
            
            critic_real = critic(real).reshape(-1)
            critic_fake = critic(fake).reshape(-1)
            
            loss_critic = -(torch.mean(critic_real) - torch.mean(critic_fake))
            
            critic.zero_grad()
            loss_critic.backward(retain_graph=True) # retain_graph=True because we use fake for generator
            opt_critic.step()

            # Clip critic weights
            for p in critic.parameters():
                p.data.clamp_(-clip_value, clip_value)

        # Train Generator: min -E[critic(fake)] <--> max E[critic(fake)]
        output = critic(fake).reshape(-1)
        loss_gen = -torch.mean(output)
        
        gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()


    print(f"Epoch [{epoch+1}/{num_epochs}] Loss C: {loss_critic:.4f}, Loss G: {loss_gen:.4f}")

    # You can reuse the visualization code from the standard GAN here!
    # Append the losses for plotting
    G_losses.append(loss_gen.item())
    D_losses.append(loss_critic.item())
    with torch.no_grad():
        # Generate images from the fixed noise vector
        fake_samples = gen(fixed_noise)
        
        # You can save the images or display them
        # Option 1: Save a grid of images
        from torchvision.utils import save_image
        save_image(fake_samples, f"gan_samples_WGAN/sample_epoch_{epoch+1}.png", normalize=True)

Epoch [1/50] Loss C: -1.1545, Loss G: 0.6121


In [ ]:
# !pip install matplotlib


In [ ]:
import matplotlib.pyplot as plt

# After the training loop finishes
plt.figure(figsize=(10,5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses, label="G")
plt.plot(D_losses, label="D")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.savefig("gan_loss_plot.png") # Save the plot to a file
plt.show()